Just some test

In [176]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [177]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

# import matplotlib
import matplotlib.pyplot as plt

root_dir = os.path.abspath('..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.encoders import Encoder
from src.data_loader.transforms_numpy import polar2cartesian, addSpeckleNoise, energyLoss, addBandReflects



## 1. Encoder test

> Test of modules from `./src/models/encoders.py`.

> Responsible for creating features map and context map. 

In [178]:
model_encoder = Encoder(in_ch = 1, out_ch = 128, dim = 32, dropout=0.5, norm_fn ='instance') # for features encoder

# model_encoder = Encoder(in_ch = 1, out_ch = 128, dim = 32, dropout=0.5, norm_fn =None) # for context encoder

In [179]:
from src.data_loader.transforms_numpy import polar2cartesian, addSpeckleNoise, energyLoss, addBandReflects

In [180]:
# example of data
data_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/seq_1/fls/352.png'
new_frame = cv2.imread(data_pth, 0)
noise_frame = addBandReflects(new_frame, omega1 = 0.03, omega2 = 0.07, gain = 0.02)
# noise_frame = energyLoss(noise_frame, alpha = 0.02)
noise_frame = addSpeckleNoise(noise_frame, m_min = 30, m_max = 100, sigma = 0.25, beam_width=2.0)
noise_frame = noise_frame.astype(np.uint8)

# print(f'sample size: sample.shape')

sample = torch.from_numpy(noise_frame.copy()) # convert to torch tensor
sample = sample.float()
sample = sample.unsqueeze(0) # add channels num dimension
sample = sample.unsqueeze(0) # add batch size dimension
sample = sample.unsqueeze(0) # add frame num inm series dimension

print('+','-'*80,'+')
print(f'input tensor shape: {sample.shape}, data type: {sample.dtype}')
print('+','-'*80,'+')

+ -------------------------------------------------------------------------------- +
input tensor shape: torch.Size([1, 1, 1, 800, 768]), data type: torch.float32
+ -------------------------------------------------------------------------------- +


In [181]:
# pass through encoder -> downsized 4 times

output = model_encoder(sample)

print('+','-'*80,'+')
print(f'output tensor shape: {output.shape}, data type: {output.dtype}')
print('+','-'*80,'+')



+ -------------------------------------------------------------------------------- +
output tensor shape: torch.Size([1, 128, 200, 192]), data type: torch.float32
+ -------------------------------------------------------------------------------- +


## 2. Patch extraction test

> Test of modules from `./src/models/patchifier.py`.

> Responsible for selection of keypoints for patches and extraction of patches from features map from coords corresponding to key points.

In [182]:
from src.models.patchifier import Patchifier

In [183]:
config_file_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'


with open(config_file_pth, "r") as f:
    config = Box(yaml.safe_load(f))

patchifier = Patchifier(config, debug_mode = True)

# del config

''' 
If debug mode, you can acces some debug data from dictionary. 

    patchifier.debug_dict['key']

Available keys:
> 'harris_response' - harris response used to find keypoints, np.array, uint8
> 'key_points' - input frame with drawn keypoints and gird


'''

print('+','-'*80,'+')
print(f'input tensor shape: {sample.shape}, data type: {sample.dtype}')
print('+','-'*80,'+')




+ -------------------------------------------------------------------------------- +
input tensor shape: torch.Size([1, 1, 1, 800, 768]), data type: torch.float32
+ -------------------------------------------------------------------------------- +


In [184]:
# pass input sample image through patchifier module

coords, patches, fmap, imap = patchifier(sample, mode = 'harris')

# get from debug dict key frames
debug_keypoints = patchifier.debug_dict['key_points']
debug_harrisresponse = patchifier.debug_dict['harris_response']

fig, ax = plt.subplots(1, 2, figsize = (15,5))
fig.suptitle('Patchifier results')

ax[0].set_title('Keypoints on grid')
ax[0].imshow(debug_keypoints)


ax[1].set_title('Harris response')
ax[1].imshow(debug_harrisresponse, cmap='hot')

plt.show()



NameError: name 'b' is not defined

In [ ]:
print('='*100)
print(f'\nFeatures & contex map:\n- feature map (fmap): size = {fmap.shape}\n- context map (imap): size = {imap.shape}')
print(f'\nPatches:\n- shape: {patches.shape}\n- number of patches: {patches.shape[2]}\n- patch size: {patches.shape[-2]}x{patches.shape[-1]}')
print(f'\nCoords:\n- shape: {coords.shape}\n- number of key points for patches: {coords.shape[1]}')
print('\n','='*100)


Features & contex map:
- feature map (fmap): size = torch.Size([1, 1, 128, 200, 192])
- context map (imap): size = torch.Size([1, 1, 128, 200, 192])

Patches:
- shape: torch.Size([1, 1, 40, 128, 3, 3])
- number of patches: 40
- patch size: 3x3

Coords:
- shape: torch.Size([1, 40, 2])
- number of key points for patches: 40



In [ ]:

coords, patches, fmap, imap = patchifier(sample, mode = 'harris')

print(f'{coords.shape}')
print(f'{patches.shape}')
print(f'{fmap.shape}')
print(f'{imap.shape}')

torch.Size([1, 40, 2])
torch.Size([1, 1, 40, 128, 3, 3])
torch.Size([1, 1, 128, 200, 192])
torch.Size([1, 1, 128, 200, 192])


# 3. Graph module test

> Test of modules from `./src/models/graph.py`.

> Responsible for operating graph and frames processing

In [ ]:
from src.models.graph import Graph

from src.data_loader.data_loader import data_headers as h # headers of csv file

In [ ]:
data_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/seq_1/fls'
data_csv_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/seq_1/sequence.csv'
config_file_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'

with open(config_file_pth, "r") as f:
    config = Box(yaml.safe_load(f))

PatchGraph = Graph(config)

In [ ]:
loop_size = 15

# dataframe for csv data
data_reader = np.loadtxt(data_csv_pth, delimiter = ',', skiprows=1) # read data from csv 

# Loop through frames
for frame_no in range(loop_size):
    frame_name = os.path.join(data_pth, f'{frame_no}.png')
    frame = cv2.imread(frame_name, 0)
    
    # add noises
    noise_frame = addBandReflects(new_frame, omega1 = 0.03, omega2 = 0.07, gain = 0.02)
    # noise_frame = energyLoss(noise_frame, alpha = 0.02)
    noise_frame = addSpeckleNoise(noise_frame, m_min = 30, m_max = 100, sigma = 0.25, beam_width=2.0)
    noise_frame = noise_frame.astype(np.uint8)

    # to tensor
    sample = torch.from_numpy(noise_frame.copy()) # convert to torch tensor
    sample = sample.float()
    sample = sample.unsqueeze(0).unsqueeze(0).unsqueeze(0) # add channels num dimension; # add batch size dimension; # add frame num in series dimension

    # get time stamp:
    time_stamp = data_reader[frame_no, h['t']]

    PatchGraph(sample, time_stamp)

    if not frame is None:   
        print(f'Frame [{frame_no}], t = {time_stamp} added to graph')



buff shape: torch.Size([36, 40, 3, 3, 128])
new patches shape: torch.Size([1, 1, 40, 128, 3, 3])


RuntimeError: The expanded size of the tensor (128) must match the existing size (3) at non-singleton dimension 4.  Target sizes: [0, 40, 3, 3, 128].  Tensor sizes: [40, 128, 3, 3]

In [ ]:
# check some data after sequence frocessing

# 1. Global graph index
print('='*40)
print(f'Check if graph global idx is correct')
print(f'Global graph idx: {PatchGraph.frame_n} (expected: {loop_size})')

# 2. Check if correct number of frames is added to frame graph 
print('='*40)
print(f'Check if correct number of frames is added ')
buff_size = config.BUFF_SIZE
error = {}
for i in range(buff_size):
    sum = torch.sum(PatchGraph.fmap[i, :, :, :]).detach().cpu().numpy()
    
    if i < loop_size:
        if sum != 0: pass # print(f'- {i} OK')
        else: 
            print(f'- {i} ERR')
            error.append(i)
    if i >= loop_size:
        if sum == 0: pass # print(f'- {i} OK')
        else: 
            print(f'- {i} ERR')
            error.append(i)

print(f'number of errors: {len(error)}')

Check if graph global idx is correct
Global graph idx: 15 (expected: 15)
Check if correct number of frames is added 
number of errors: 0
